# Spotify Popularity and Genre Modelling

This notebook summarises a public-safe machine-learning workflow for two tasks:

1. predicting a numeric song popularity score;
2. predicting the top genre label for a song.

Raw CSV files are not included in the repository. The reusable implementation lives in `src/spotify_ml/` and can be run from the command line after adding the four CSV files documented in `data/README.md`.

## 1. Setup

The notebook reads the pre-generated public report assets where possible. If raw data is added locally, the same workflow can be rerun using the command-line runner.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path('..')
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = PROJECT_ROOT / 'figures'
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

metrics_path = REPORTS_DIR / 'local_validation_metrics.csv'
metrics = pd.read_csv(metrics_path)
metrics

## 2. Local validation results

The validation split is fixed with `RANDOM_STATE = 42`. Regression uses RMSE, where lower is better. Classification uses accuracy, where higher is better.

In [ ]:
regression_metrics = metrics[metrics['task'] == 'regression'].sort_values('validation_metric')
classification_metrics = metrics[metrics['task'] == 'classification'].sort_values('validation_metric', ascending=False)

print('Regression models sorted by validation RMSE:')
display(regression_metrics)

print('Classification models sorted by validation accuracy:')
display(classification_metrics)

## 3. Figures

These figures were generated by the experiment runner. They are kept in the repository because they do not expose raw data rows.

In [ ]:
for figure_name in [
    'regression_target_distribution.png',
    'regression_validation_rmse.png',
    'classification_class_distribution.png',
    'classification_validation_accuracy.png',
]:
    path = FIGURES_DIR / figure_name
    print(path.name)
    display(Image(filename=str(path)))

## 4. Model choice

### Regression

The final regression workflow uses CatBoost because it handles high-cardinality categorical features such as artist, genre and decade without manual one-hot expansion. The repository exposes both a quick mode and a fuller mode.

### Classification

The final classification workflow uses a balanced Linear SVC over numeric features and one-hot encoded artist/decade features. The optional TF-IDF text variant is not the default because it performed worse on the reproducible validation split.

## 5. Rerunning locally

After placing the four required CSV files in `data/raw/`, run:

```bash
python -m spotify_ml.run_experiments --data-dir data/raw --output-dir outputs --figure-dir figures
```

A slower regression run with title text enabled can be run with:

```bash
python -m spotify_ml.run_experiments --mode full --use-title-text
```

In [ ]:
required_files = [
    'spotify_regression_train.csv',
    'spotify_regression_test.csv',
    'spotify_classification_train.csv',
    'spotify_classification_test.csv',
]
missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing:
    print('Raw data is not included in the public repository.')
    print('Add the files listed in data/README.md to rerun the full experiment.')
else:
    print('All raw data files are present. You can rerun the full experiment from the terminal.')

## 6. Limitations

The dataset is small, the classification target has a long tail of rare genre labels, and the public benchmark split is different from the local validation split. The validation results should therefore be interpreted as model-selection evidence rather than as exact estimates of hidden benchmark performance.